# Energy Transformers: ListOps Model Combinations on Cloud GPUs

This notebook runs `listops_model_combinations.py` on Google Colab or Kaggle Notebooks with automatic platform detection.

**Prerequisites:**
- W&B API key (you'll be prompted to login)
- WANDB_ENTITY environment variable set before running the script
- GPU runtime enabled (set in notebook settings)

## 1. Platform Detection & Setup

In [ ]:
import os
import sys
from pathlib import Path


# Platform detection
def detect_platform():
    """Detect if running on Colab, Kaggle, or local."""

    if os.path.exists("/kaggle"):
        return "kaggle"

    try:
        # Note: kaggle also supports google.colab!
        # Thats why we check for kaggle first, then colab.
        import google.colab

        return "colab"
    except ImportError:
        pass
    return "local"


platform = detect_platform()
print(f"🖥️  Detected platform: {platform.upper()}")

# Setup paths based on platform
if platform == "colab":
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    repo_dir = "/content/energy-transformers-tcrp"
    output_base = "/content/drive/MyDrive/energy-transformers-experiments"
elif platform == "kaggle":
    repo_dir = "/kaggle/working/energy-transformers-tcrp"
    output_base = "/kaggle/working/outputs"
else:
    repo_dir = os.getcwd()  # Assume already in repo
    output_base = os.path.join(repo_dir, "outputs")

print(f"📂 Repo directory: {repo_dir}")
print(f"💾 Output base: {output_base}")

🖥️  Detected platform: LOCAL
📂 Repo directory: /home/luisegzb/Development/energy-transformers-tcrp/notebooks
💾 Output base: /home/luisegzb/Development/energy-transformers-tcrp/notebooks/outputs


## 2. Clone Repository (if needed)

In [ ]:
if platform != "local":
    if not os.path.exists(repo_dir):
        print(f"📥 Cloning repository to {repo_dir}...")
        os.system(
            f"git clone https://github.com/luisemilio14/energy-transformers-tcrp.git {repo_dir}"
        )
    else:
        print(f"✅ Repository already exists at {repo_dir}")
else:
    print("ℹ️  Running locally, assuming repo is current directory.")

ℹ️  Running locally, assuming repo is current directory.


## 3. Navigate to Repo & Install Dependencies

In [ ]:
os.chdir(repo_dir)
print(f"📍 Current directory: {os.getcwd()}")

# Install dependencies
print("\n📦 Installing dependencies...")
!pip install poetry


📍 Current directory: /home/luisegzb/Development/energy-transformers-tcrp/notebooks

📦 Installing dependencies...


2

In [ ]:
!poetry install

## 4. Authenticate with Weights & Biases

In [ ]:
from kaggle_secrets import UserSecretsClient

wandb_api_key = UserSecretsClient().get_secret("WANDB_API_KEY")
wandb_entity = UserSecretsClient().get_secret("WANDB_ENTITY")

In [ ]:
import wandb

print("🔐 Authenticating with Weights & Biases...")
wandb.login(key=wandb_api_key)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


🔐 Authenticating with Weights & Biases...


wandb: Currently logged in as: luisegzb (luisegzb-universidade-federal-de-minas-gerais) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 5. Set Environment Variables

In [ ]:
os.environ["WANDB_ENTITY"] = wandb_entity
print("✅ Set WANDB_ENTITY")

ValueError: WANDB_ENTITY cannot be empty!

## 6. Prepare Output Directory

In [ ]:
# Create output directory
Path(output_base).mkdir(parents=True, exist_ok=True)
print(f"✅ Output directory ready: {output_base}")

# For local/Kaggle, also set an environment variable for the script if needed
os.environ["OUTPUT_DIR"] = output_base

✅ Output directory ready: /home/luisegzb/Development/energy-transformers-tcrp/notebooks/outputs


## 7. Run the Model Combinations Script

In [ ]:
!poetry run dvc repro listops32_tokenize

In [ ]:
import subprocess

print("\n" + "=" * 70)
print("🚀 Starting model combinations training...")
print("=" * 70)

# Run the script
script_path = os.path.join(repo_dir, "src", "listops_parallel_model_comb.py")

if not os.path.exists(script_path):
    raise FileNotFoundError(f"Script not found at {script_path}")

# Add repo src to path
sys.path.insert(0, os.path.join(repo_dir, "src"))

# Run script
try:
    result = subprocess.run(
        [sys.executable, script_path, "--config=params.yaml"],
        cwd=repo_dir,
        capture_output=False,
        text=True,
    )

    if result.returncode == 0:
        print("\n" + "=" * 70)
        print("✅ Script completed successfully!")
        print("=" * 70)
    else:
        print(f"\n❌ Script exited with code {result.returncode}")

except Exception as e:
    print(f"\n❌ Error running script: {e}")
    raise

## 8. Post-Execution Summary

In [ ]:
print("\n📋 Execution Summary:")
print(f"  Platform: {platform.upper()}")
print(f"  W&B Entity: {os.environ.get('WANDB_ENTITY', 'NOT SET')}")
print(f"  Output Directory: {output_base}")

# List output files if any
if os.path.exists(output_base):
    files = list(Path(output_base).glob("**/*"))[:10]  # First 10 items
    if files:
        print(f"\n  Output files ({len(files)} shown):")
        for f in files:
            if f.is_file():
                print(f"    - {f.relative_to(output_base)}")

if platform == "colab":
    print(
        f"\n💾 Results are saved to Google Drive: /MyDrive/energy-transformers-experiments"
    )
    print("   Access them via Files panel on the left.")
elif platform == "kaggle":
    print(f"\n💾 Results are saved to: /kaggle/working/outputs")
    print("   Download them from the Output section after notebook completes.")
else:
    print(f"\n💾 Results are saved locally to: {output_base}")